# Семинар 2. Как учат языковую модель

Первый семинар был про готовую модель: следующий токен и примеры в промпте. Здесь — откуда берутся веса.

Три ступени из лонгрида: предобучение, SFT, затем предпочтения людей (RLHF или DPO). Веса в ноутбуке не обновляем. Считаем ту функцию, которую при обучении уменьшают.

Та же среда, что в первом семинаре: `.venv` и Ollama.

## Предобучение

Берут сырой текст. Метка для каждого шага — следующее слово в этом тексте, её не размечают руками.

$$
L_1 = \sum_i \log P(u_i \mid u_{i-k}, \ldots, u_{i-1})
$$

Модель не видит текущий токен, когда его предсказывает. Иначе она списала бы ответ. Ниже для фразы напечатаны пары «уже прочитано → что надо угадать», потом средний логарифм вероятности. У связного предложения он выше, чем у тех же слов вразнобой.

In [ ]:
import math

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

logging.set_verbosity_error()

tokenizer = AutoTokenizer.from_pretrained("openai-community/openai-gpt")
gpt1 = AutoModelForCausalLM.from_pretrained("openai-community/openai-gpt")
gpt1.eval()


def token_logprobs(text):
    ids = tokenizer(text, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        logits = gpt1(ids).logits
    log_probs = torch.log_softmax(logits[0, :-1], dim=-1)
    target = ids[0, 1:]
    chosen = log_probs[torch.arange(target.shape[0]), target]
    return chosen


text = "once upon a time there was a king"
pieces = [tokenizer.decode([i]) for i in tokenizer(text)["input_ids"]]
for i in range(1, len(pieces)):
    print(" | ".join(pieces[:i]), "->", pieces[i])

order = token_logprobs(text)
shuffled = token_logprobs("king a was there time upon once")
print("связный текст, средний log", round(float(order.mean()), 3))
print("те же слова вразнобой, средний log", round(float(shuffled.mean()), 3))

## Разрыв

Цель предобучения — угадать продолжение книги. Человек хочет ответ на вопрос. Это разные задачи.

<img src="img/gpt3_vs_instruct.png" width="900">

Слева GPT-3 дописывает вопрос вариантами A–D. Справа та же модель после выравнивания объясняет, что делает список.

<img src="img/three_h.png" width="560">

От выровненной модели ждут три вещи сразу: полезность, безвредность, честность. Угол между ними — где модель либо отказывается отвечать, либо врёт, либо помогает сделать вредное.

In [ ]:
def next_prob(prompt, word):
    prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]
    token_id = tokenizer(prompt + word)["input_ids"][-1]
    with torch.no_grad():
        logits = gpt1(prompt_ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    return float(probs[token_id])


prompt = "the capital of france is"
p_a = next_prob(prompt, " a")
p_paris = next_prob(prompt, " paris")
print("P(a)", round(p_a, 4))
print("P(paris)", round(p_paris, 4))
print("во сколько раз a вероятнее", round(p_a / p_paris))

`a` вероятнее `paris` больше чем в сто раз. Предобученная модель честно продолжает текст. Короткий правильный ответ для неё редкий. SFT как раз поднимает вероятность таких ответов.

## SFT

Человек пишет ответ, который модель должна была выдать. Пара «промпт, ответ» — одна строка обучения. В статье InstructGPT таких пар около 13 тысяч.

Любую задачу записывают как продолжение текста. Одна и та же голова предсказывает следующий токен и на предобучении, и здесь.

<img src="img/sft_templates.png" width="900">

$$
L_2 = \sum \log P(y \mid x)
$$

Справа на картинке ниже — какие промпты вообще просили у модели. Почти половина — «напиши текст», не классификация.

<img src="img/sft_prompts.png" width="820">

In [ ]:
prompt_sft = "Объясни, что такое токен, одним предложением."
good_answer = "Токен — это кусок текста, который модель читает как одну единицу."

print("промпт:", prompt_sft)
print("ответ человека:", good_answer)
print("это одна строка для L2")

В ячейке выше можно заменить `good_answer` на свой. Обучение от этого не запустится: мы только собираем строку датасета.

## RLHF

SFT учит повторять записанные ответы. Дальше люди сравнивают несколько ответов на один промпт, а не пишут идеал с нуля.

<img src="img/rlhf_steps.png" width="980">

Шаг 1 — те самые пары SFT. Шаг 2 — на один промпт несколько ответов, человек ставит их по порядку, из этого учат модель награды. Шаг 3 — языковая модель генерирует новый ответ, награда говорит, хорош ли он, и веса сдвигают в эту сторону. Алгоритм шага 3 называется PPO.

Модель награды получает промпт и ответ и выдаёт одно число. Её учат ставить победителю больше, чем проигравшему:

$$
L_{RM} = -\mathbb{E}\,\log \sigma\big(r(x, y_w) - r(x, y_l)\big)
$$

$y_w$ — ответ, который человек поставил выше, $y_l$ — который ниже. $\sigma$ переводит разницу оценок в вероятность от 0 до 1. Если хороший ответ оценён выше, штраф маленький. Если модель награды перепутала порядок, штраф большой.

In [ ]:
def pair_loss(score_good, score_bad):
    margin = score_good - score_bad
    return -math.log(1 / (1 + math.exp(-margin)))


print("хороший выше", round(pair_loss(2.0, 0.5), 3))
print("порядок перепутан", round(pair_loss(0.5, 2.0), 3))

Без тормоза модель быстро находит дыру в награде и пишет странный текст, который та всё равно высоко оценивает. Из награды вычитают, насколько новая модель уехала от SFT-модели:

$$
-\beta \log \frac{\pi_{RL}(y \mid x)}{\pi_{SFT}(y \mid x)}
$$

Ноль, если вероятности совпали. Большой штраф, если новый ответ для старой модели был почти невозможен. Отдельно небольшой добавкой модель продолжают учить на обычном тексте, чтобы она не разучилась строить фразы.

In [ ]:
# ответ стал в 10 раз вероятнее, чем у SFT-модели
print("log(10) =", round(math.log(10), 3))
print("штраф при beta = 0.1:", round(0.1 * math.log(10), 3))

Так выглядит сбор сравнений в продукте: два ответа, человек отмечает лучший. Эти пары и есть датасет шага 2.

<img src="img/preference_ui.png" width="760">

Ниже Qwen пишет два ответа на один промпт. Выберите лучший в следующей ячейке: поставьте `winner = 1` или `winner = 2`.

In [ ]:
import requests

MODEL = "qwen2.5:3b"
OLLAMA = "http://localhost:11434"
prompt = "Объясни пятикласснику, что такое токен. Два предложения."


def generate(prompt, seed):
    response = requests.post(
        OLLAMA + "/api/generate",
        json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False,
            "raw": True,
            "keep_alive": "30m",
            "options": {"temperature": 0.8, "num_predict": 80, "seed": seed},
        },
        timeout=180,
    )
    response.raise_for_status()
    return response.json()["response"]


tags = requests.get(OLLAMA + "/api/tags", timeout=5)
tags.raise_for_status()
names = [model["name"] for model in tags.json()["models"]]
if not any(name.startswith(MODEL) for name in names):
    print("скачиваю", MODEL)
    pulled = requests.post(
        OLLAMA + "/api/pull",
        json={"model": MODEL, "stream": False},
        timeout=1800,
    )
    pulled.raise_for_status()
    print(pulled.json()["status"])

answers = []
for seed in (1, 2):
    text = generate(prompt, seed)
    answers.append(text)
    print("ответ", seed)
    print(text)
    print()

In [ ]:
winner = 1
loser = 2 if winner == 1 else 1
print("лучше, ответ", winner)
print(answers[winner - 1])
print()
print("хуже, ответ", loser)
print(answers[loser - 1])
print()
print("пара (промпт, yw, yl) готова и для модели награды, и для DPO")

## DPO

Тот же датасет пар, без отдельной модели награды и без цикла PPO. Языковая модель сама себе награда: хороший ответ должен стать вероятнее, чем у замороженной SFT-модели, плохой — менее вероятным.

<img src="img/dpo_vs_rlhf.png" width="900">

$$
L_{DPO} = -\mathbb{E}\,\log \sigma \Big(
\beta \log \frac{\pi(y_w \mid x)}{\pi_{ref}(y_w \mid x)}
-
\beta \log \frac{\pi(y_l \mid x)}{\pi_{ref}(y_l \mid x)}
\Big)
$$

$\pi_{ref}$ — модель до этого шага. $\beta$ обычно 0.1–0.5: больше $\beta$ — меньше разрешённый отход от неё.

Пока $\pi$ и $\pi_{ref}$ — одна и та же модель, оба логарифма отношения равны нулю, учиться не из чего. Ниже числа GPT-1 для `paris` и `a`. Потом подставляем воображаемый шаг: хороший ответ стал в десять раз вероятнее, плохой — в десять раз менее вероятным. Веса не меняются, мы только считаем скобку формулы.

In [ ]:
beta = 0.1
log_ref_good = math.log(p_paris)
log_ref_bad = math.log(p_a)


def dpo_margin(log_new_good, log_ref_good, log_new_bad, log_ref_bad):
    good_term = log_new_good - log_ref_good
    bad_term = log_new_bad - log_ref_bad
    return beta * (good_term - bad_term)


def dpo_loss(margin):
    return -math.log(1 / (1 + math.exp(-margin)))


before = dpo_margin(log_ref_good, log_ref_good, log_ref_bad, log_ref_bad)
after = dpo_margin(
    log_ref_good + math.log(10),
    log_ref_good,
    log_ref_bad - math.log(10),
    log_ref_bad,
)
print("скобка до шага", round(before, 3), "loss", round(dpo_loss(before), 3))
print("скобка после воображаемого шага", round(after, 3), "loss", round(dpo_loss(after), 3))

## Что из этого вышло

<img src="img/likert.png" width="640">

Шкала от 1 до 7, выше лучше. Обычный GPT ниже, чем GPT с промптом. SFT выше промпта. PPO ещё выше. FLAN и T0 — другие способы учить модель следовать инструкции, на них можно не останавливаться.

Отдельно люди в 85% случаев предпочли InstructGPT на 1.3B параметров обычному GPT-3 на 175B. Выравнивание здесь перевесило размер.

## Что увидели

Предобучение поднимает вероятность настоящего продолжения текста. У связной фразы логарифм выше, чем у перемешанных слов. Правильный короткий ответ при этом может быть редким: `paris` против `a`.

SFT — это пары, которые пишет человек. RLHF добавляет сравнение нескольких ответов и модель награды. DPO делает то же сравнение одной функцией потерь, без награды и без PPO.

Пара, которую вы отметили у Qwen, и есть одна строка такого датасета.